# simulation.ipynb

Scores a fitted model against the **full** choice set: every PUMS unit in `pums_file` is given a
utility for staying and for each of the ~2.3k destination PUMAs, using the coefficients in a larch
`*_spec.yaml`. Destination-side data is rebuilt straight from the per-geography census tables, so no
`estdata` parquet (with its 100 sampled alternatives) is involved -- and with the whole choice set
present there is no sampling correction to add.

The utility functions are read off `lib.model_spec`, so they track the estimated model as it changes;
only the handful of terms that `build_long_data` computes ad-hoc (distance, same-state/CBSA, ...) are
spelled out here.

Outputs, both indexed origin MIGPUMA x destination PUMA:

- `utility_matrix` -- `PERWT`-weighted mean destination utility of an origin's residents. Cells whose
  PUMA lies in the origin MIGPUMA itself are `NaN`: that region is the stay alternative, not a move.
- `flow_matrix` -- expected movers, i.e. `PERWT`-weighted choice probabilities.

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

sys.path.insert(0, os.path.abspath(".."))
from lib import census as lc
from lib import io as lio
from lib import model_spec as lm
from lib import modeling_util as lut

JAX not found. Some functionality will be unavailable.


In [2]:
year = 2018
data_dir = Path("../data")
pums_file = data_dir / f"pums/pums_100_{year}.parquet"
spec_file = Path("results/us_mnl_estdata_pums_100_2018_100_1786120685_spec.yaml")

# Every unit is scored against all ~2.3k PUMAs, so cost is (rows x PUMAs x terms) -- about
# 2 s per 5k units, i.e. ~10 min for the whole 1.7M-row file. Subsample to keep this
# interactive; None uses every row. Flows are scaled back up by 1 / sample_frac below.
sample_frac = 0.05
batch_size = 5_000
seed = 42

### Coefficients

In [3]:
weights = lut.extract_weights(spec_file)
pd.Series(weights)

destchoice_T34                       -0.097814
destchoice_alt_commute               -0.691849
destchoice_birthstate                 0.891646
destchoice_logdist                   -0.873439
destchoice_med_earnings_10k_degree    0.212368
                                        ...   
stay_metro                           -0.059941
stay_mil                             -2.118067
stay_recently_divorced_or_widowed    -0.554920
stay_single_parent                    0.138363
stay_vacancy_rate                     0.857699
Length: 61, dtype: float64

### Data

In [4]:
pums = pd.read_parquet(pums_file)
if sample_frac is not None:
    pums = pums.sample(frac=sample_frac, random_state=seed)

puma_data, migpuma_data = lc.load_geo_tables(data_dir, year)

# the distance matrix defines the geography on both sides: its rows are the origin MIGPUMAs,
# its columns the full destination choice set
distance_matrix = lio.load_distance_matrix(
    data_dir / "distances/puma_migpuma_distance_matrix.csv", zfill=7
)
puma_migpuma = lio.load_puma_migpuma(
    data_dir / "geometry/equivalencies/puma_migpuma_2010.csv"
)

dest_pumas = distance_matrix.columns
origin_migpumas = distance_matrix.index
n_dest = len(dest_pumas)
len(pums), n_dest, len(origin_migpumas)

(86952, 2336, 975)

### Origin side

The origin MIGPUMA's census row joined onto each unit as `.ORIG`, plus the individual-level columns
`create_estdata.ipynb` derives on its way to an estdata parquet (the pums file stops before them).

In [5]:
# only the origin columns some term actually reads -- the census tables are ~2k columns
# wide, which would dwarf the pums file itself. The .ORIG columns not found here are the
# OWN_* ones derived a few cells down.
origin_cols = {
    c.removesuffix(".ORIG")
    for c in lm.required_individual_columns()
    if c.endswith(".ORIG")
}
df = pums.join(
    migpuma_data[sorted(origin_cols & set(migpuma_data.columns))].add_suffix(".ORIG"),
    on="ORIGIN",
    how="left",
)
assert df["TOT_POP.ORIG"].notna().all(), "some ORIGIN is missing from the MIGPUMA table"
n = len(df)

# rows of the distance matrix double as the origin axis of every output below
origin_row = origin_migpumas.get_indexer(df["ORIGIN"])
assert (origin_row >= 0).all()

In [6]:
# NAICS skill/credential group dummies. Active-duty members are excluded from every group
# (they carry their own military term instead), so they have no group at all.
for sfx in lc.UNIT_SUFFIXES:
    sector = df[f"INDNAICS_{sfx}"].str[:2].map(lc.NAICS_SECTOR_PREFIXES)
    sector = sector.where(~df[f"IN_MILITARY_{sfx}"].astype(bool))
    for group, members in lc.NAICS_GROUPS.items():
        df[f"NAICS_{group}_{sfx}"] = sector.isin(members).astype(np.int8)

In [7]:
# Each unit's own category as a position into the matching per-geography column list, used
# below to gather "the area's share of people like me". -1 means no category, which drops
# the term rather than scoring category 0.
race_cols = list(dict.fromkeys(lc.RACE_COLUMNS.values()))
edu_cols = list(lc.EDU_EARNINGS_RAW_COLS.values())

age_flags = df[lc.AGE_BRACKET_FLAGS].to_numpy()
edu_flags = df[list(lc.EDU_EARNINGS_RAW_COLS)].to_numpy()
assert (age_flags.sum(axis=1) == 1).all(), "AGE_* dummies aren't exclusive/exhaustive"
assert (edu_flags.sum(axis=1) == 1).all(), "EDU_* dummies aren't exclusive/exhaustive"
age_code = age_flags.argmax(axis=1)
edu_code = edu_flags.argmax(axis=1)

race_code, naics_code = {}, {}
for sfx in lc.UNIT_SUFFIXES:
    names = df[f"RACE_ETHNICITY_{sfx}"].map(lc.RACE_COLUMNS).fillna("")
    race_code[sfx] = pd.Index(race_cols).get_indexer(names)
    groups = df[[f"NAICS_{g}_{sfx}" for g in lc.NAICS_GROUPS]].to_numpy()
    naics_code[sfx] = np.where(groups.any(axis=1), groups.argmax(axis=1), -1)

In [8]:
# Per-member weight for the paired race/NAICS terms: 1 / (members in a category that
# carries a coefficient). A flat 0.5 would halve a couple whose only contributing member is
# one of the two slots, since _SEC duplicates _REF for unpaired units.
modeled_factors = {f for s in lm.SHARED_SPECS for f in s.factors if isinstance(f, str)}
for kind, categories, codes in (
    ("RACE", lc.RACE_CATEGORIES, race_code),
    ("NAICS", [f"NAICS_{g}" for g in lc.NAICS_GROUPS], naics_code),
):
    contributes = {}
    for sfx in lc.UNIT_SUFFIXES:
        modeled = [c for c in categories if f"{c}_{sfx}" in modeled_factors]
        assert modeled, f"no {kind} category has a SharedSpec"
        in_modeled = df[[f"{c}_{sfx}" for c in modeled]].to_numpy().max(axis=1) > 0
        contributes[sfx] = in_modeled & (codes[sfx] >= 0)
    n_present = sum(c.astype(np.int8) for c in contributes.values())
    for sfx in lc.UNIT_SUFFIXES:
        df[f"{kind}_W_{sfx}"] = np.where(
            contributes[sfx], 1.0 / np.maximum(n_present, 1), 0.0
        )

In [9]:
# Origin-side counterparts of the per-destination OWN_* columns, kept per member so that a
# term pairs a member's category indicator with that same member's share.
origin_pos = migpuma_data.index.get_indexer(df["ORIGIN"])
assert (origin_pos >= 0).all()
rows = np.arange(n)


def gather_origin(cols, code):
    values = migpuma_data[cols].to_numpy(np.float64)[origin_pos]
    return np.where(code < 0, 0.0, values[rows, np.maximum(code, 0)])


for sfx in lc.UNIT_SUFFIXES:
    df[f"OWN_RACE_ETH_PROP_{sfx}.ORIG"] = gather_origin(race_cols, race_code[sfx])
    df[f"OWN_NAICS_GROUP_PROP_{sfx}.ORIG"] = gather_origin(
        lc.NAICS_GROUP_PROP_COLUMNS, naics_code[sfx]
    )
df["OWN_EARNINGS_10K_BY_EDU.ORIG"] = gather_origin(edu_cols, edu_code)

In [10]:
individual_cols = sorted(lm.required_individual_columns())
missing = [c for c in individual_cols if c not in df.columns]
assert not missing, missing
na = df[individual_cols].isna().sum()
assert na.sum() == 0, na[na > 0]

### Destination side

One row per PUMA in the choice set. Most `ALT{i}_<suffix>` columns of an estdata parquet are a plain
per-PUMA value; the `OWN_*` ones are the PUMA's share of people in the *unit's own* category, so they
become a (PUMA x category) table indexed by the codes above.

In [11]:
dest = puma_data.loc[dest_pumas]
plain = {
    suffix: dest[col].to_numpy(np.float64) for suffix, col in lc.CENSUS_FIELDS.items()
}
gathers = {
    "OWN_AGE_PROP": (dest[lc.AGE_BRACKET_COLS].to_numpy(np.float64), age_code),
    "OWN_EARNINGS_10K_BY_EDU": (dest[edu_cols].to_numpy(np.float64), edu_code),
    **{
        f"OWN_RACE_ETH_PROP_{s}": (dest[race_cols].to_numpy(np.float64), race_code[s])
        for s in lc.UNIT_SUFFIXES
    },
    **{
        f"OWN_NAICS_GROUP_PROP_{s}": (
            dest[lc.NAICS_GROUP_PROP_COLUMNS].to_numpy(np.float64),
            naics_code[s],
        )
        for s in lc.UNIT_SUFFIXES
    },
}

puma_type = dest["TYPE_NUM"].to_numpy()
puma_cbsa = dest["NAME_NUM"].to_numpy()
puma_state = puma_migpuma.loc[dest_pumas, "State"].astype(int).to_numpy()
dest_migpuma = puma_migpuma.loc[dest_pumas, "MIGPUMA"].to_numpy()
dist = distance_matrix.to_numpy()

# TYPE/CBSA/STATE/DIST are the structural ones handled by name just above
assert lm.required_alt_suffixes() <= set(plain) | set(gathers) | {
    "TYPE",
    "CBSA",
    "STATE",
    "DIST",
}
for name, values in [*plain.items(), *((k, v[0]) for k, v in gathers.items())]:
    assert not np.isnan(values).any(), name

### Utility functions

`STAY_ONLY_SPECS` / `SHARED_SPECS` / `MOVE_ONLY_SPECS` are evaluated generically; the rest of
`MOVE_ONLY_TERMS` and `STAY_ONLY_TERMS` are the ad-hoc geography comparisons. Specs sharing a name
(the per-member race/NAICS pairs) sum into their one coefficient, as they do in training.

In [12]:
def scale(factors, col):
    """Product of a spec's unit-side factors, or None when it has none."""
    if not factors:
        return None
    out = 1.0
    for factor in factors:
        out = out * (col(factor) if isinstance(factor, str) else factor)
    return out


dest_specs = [
    (s.name, s.alt_suffix, s.factors)
    for s in lm.SHARED_SPECS + lm.MOVE_ONLY_SPECS
]
assert not any(
    "ALT_" in f for _, _, factors in dest_specs for f in factors if isinstance(f, str)
), "a destination-varying factor needs a per-PUMA lookup, not a unit-side scale"

# terms whose value is the same for every unit, so they can be summed once per PUMA
static = [t for t in dest_specs if not t[2] and t[1] not in gathers]
per_unit = [t for t in dest_specs if t not in static]

base = weights["log_pop_offset"] * np.log1p(plain["TOT_POP"])
for name, suffix, _ in static:
    base += weights[name] * plain[suffix]
base += weights["destchoice_T34"] * (puma_type == 0)
base += weights["destchoice_metro"] * (puma_type == 1)

needed = (
    {name for name, _, _ in dest_specs}
    | {s.name for s in lm.STAY_ONLY_SPECS}
    | set(lm.STAY_ONLY_TERMS)
    | set(lm.MOVE_ONLY_TERMS)
    | {"log_pop_offset"}
)
assert not needed - weights.keys(), needed - weights.keys()

In [13]:
def stay_utility(cur):
    """Utility of staying put, one value per unit in `cur`."""

    def col(name):
        return cur[name].to_numpy()

    u = weights["log_pop_offset"] * np.log1p(col("TOT_POP.ORIG"))
    for spec in lm.STAY_ONLY_SPECS:
        value = 1.0 if spec.source is None else col(spec.source)
        factor = scale(spec.factors, col)
        u = u + weights[spec.name] * value * (1.0 if factor is None else factor)
    origin_type = col("TYPE_NUM.ORIG")
    u = u + weights["stay_T34"] * (origin_type == 0)
    u = u + weights["stay_metro"] * (origin_type == 1)
    for spec in lm.SHARED_SPECS:
        factor = scale(spec.factors, col)
        value = col(spec.origin_col)
        u = u + weights[spec.name] * value * (1.0 if factor is None else factor)
    return u


def destination_utility(cur, sl):
    """(units x PUMAs) utility of moving to each PUMA in the choice set."""

    def col(name):
        return cur[name].to_numpy()

    u = np.tile(base, (len(cur), 1))
    for name, suffix, factors in per_unit:
        if suffix in gathers:
            table, code = gathers[suffix]
            code = code[sl]
            value = np.where(code[:, None] < 0, 0.0, table[:, np.maximum(code, 0)].T)
        else:
            value = plain[suffix][None, :]
        factor = scale(factors, col)
        u += weights[name] * value * (1.0 if factor is None else factor[:, None])

    u += weights["destchoice_logdist"] * np.log1p(dist[origin_row[sl]])
    u += weights["destchoice_samecbsa"] * (col("NAME_NUM.ORIG")[:, None] == puma_cbsa)
    u += weights["destchoice_samestate"] * (col("ORIGIN_STATE")[:, None] == puma_state)
    u += weights["destchoice_birthstate"] * (col("BPL_REF")[:, None] == puma_state)
    origin_type = col("TYPE_NUM.ORIG")
    # unknown type codes fall through as 0, matching training
    coded = (origin_type >= 0) & (origin_type <= 2)
    u += weights["destchoice_same_cbsa_type"] * (
        (origin_type[:, None] == puma_type) & coded[:, None]
    )
    return u

### Scoring

Utilities are accumulated per origin MIGPUMA on the fly -- a (units x PUMAs) matrix for the whole
file would not fit in memory. The `PERWT`-weighted sum over a batch's units is a one-hot matmul.

In [14]:
# a PUMA inside the unit's own origin MIGPUMA is not a move alternative -- that region is
# the stay option, so it is excluded from the choice set (as it is when sampling estdata)
own_migpuma = dest_migpuma[None, :] == origin_migpumas.to_numpy()[:, None]

n_origins = len(origin_migpumas)
utility_sum = np.zeros((n_origins, n_dest))
flow_sum = np.zeros((n_origins, n_dest))
perwt_sum = np.zeros(n_origins)
stay_sum = np.zeros(n_origins)

for start in tqdm(range(0, n, batch_size)):
    sl = slice(start, min(start + batch_size, n))
    cur = df.iloc[sl]
    perwt = cur["PERWT"].to_numpy()
    origins = origin_row[sl]

    u_dest = destination_utility(cur, sl)
    u_stay = stay_utility(cur)
    assert np.isfinite(u_dest).all() and np.isfinite(u_stay).all()

    # logit over the stay option plus every available PUMA, shifted by the row max
    available = np.where(own_migpuma[origins], -np.inf, u_dest)
    shift = np.maximum(u_stay, available.max(axis=1))
    exp_dest = np.exp(available - shift[:, None])
    exp_stay = np.exp(u_stay - shift)
    denominator = exp_stay + exp_dest.sum(axis=1)
    p_dest = exp_dest / denominator[:, None]
    p_stay = exp_stay / denominator

    # (origins x units) weight matrix: left-multiplying sums each origin's units, weighted
    spread = np.zeros((n_origins, len(cur)))
    spread[origins, np.arange(len(cur))] = perwt
    utility_sum += spread @ u_dest
    flow_sum += spread @ p_dest
    perwt_sum += np.bincount(origins, weights=perwt, minlength=n_origins)
    stay_sum += np.bincount(origins, weights=perwt * p_stay, minlength=n_origins)

  0%|          | 0/18 [00:00<?, ?it/s]

  6%|▌         | 1/18 [00:02<00:36,  2.13s/it]

 11%|█         | 2/18 [00:04<00:33,  2.06s/it]

 17%|█▋        | 3/18 [00:06<00:30,  2.05s/it]

 22%|██▏       | 4/18 [00:08<00:28,  2.04s/it]

...

 83%|████████▎ | 15/18 [00:30<00:05,  1.99s/it]

 89%|████████▉ | 16/18 [00:32<00:03,  1.99s/it]

 94%|█████████▍| 17/18 [00:34<00:02,  2.03s/it]

100%|██████████| 18/18 [00:35<00:00,  1.64s/it]

100%|██████████| 18/18 [00:35<00:00,  1.96s/it]

In [15]:
scale_up = 1.0 if sample_frac is None else 1.0 / sample_frac
scored = perwt_sum > 0

mean_utility = np.full_like(utility_sum, np.nan)
np.divide(utility_sum, perwt_sum[:, None], out=mean_utility, where=scored[:, None])

utility_matrix = pd.DataFrame(
    mean_utility, index=origin_migpumas, columns=dest_pumas
).mask(own_migpuma)
flow_matrix = pd.DataFrame(
    flow_sum * scale_up, index=origin_migpumas, columns=dest_pumas
)
utility_matrix

,0600105,0600102,0608502,0600108,0600107,0600101,0600110,0608504,0600109,0600106,...,2602200,2600900,4702402,4702300,4702401,4700500,4702501,5310400,5310300,5310100
0600100,NaN,NaN,6.413337,NaN,NaN,NaN,NaN,6.404730,NaN,NaN,...,-0.151518,-0.055036,0.265418,-0.035087,0.093903,0.281021,0.130338,0.569018,0.933123,1.475072
0600700,3.850448,3.763478,4.279991,3.564132,3.629242,4.206546,4.414777,3.850486,3.737449,3.439000,...,0.506530,0.601495,0.884960,0.706284,0.738930,1.019837,0.535928,1.220396,1.681450,2.353283
0601100,3.321344,3.397601,3.463948,2.898940,3.114157,3.934822,3.756564,3.018761,3.180881,3.007927,...,0.167730,0.207115,0.222205,-0.128962,-0.179739,0.215139,-0.143628,1.211675,1.169251,1.735744
0601300,5.870783,5.894992,5.530642,5.565291,5.605864,6.049629,6.217000,5.285063,5.923766,5.493441,...,0.113711,0.227564,0.478998,0.213057,0.314241,0.532864,0.284501,0.812003,1.145267,1.716418
0601500,3.161717,3.158499,3.582486,2.714980,2.827675,3.575294,3.725204,3.005706,3.067837,2.841229,...,0.225808,0.255364,0.616105,0.402250,0.217611,0.762128,0.198274,1.215628,1.554559,2.213645
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304007,0.069788,0.013439,0.282077,0.081208,0.035052,0.064381,0.523038,0.030208,0.310691,-0.274748,...,0.706948,0.756251,2.194670,1.732276,1.968041,2.001867,2.009437,-0.440078,-0.208035,0.375734
5151000,-0.588177,-0.715195,-0.130591,-0.904448,-0.841465,-0.442955,0.002587,-0.604471,-0.656470,-0.905955,...,1.464376,1.417859,2.066633,2.012022,1.908585,2.288854,1.795369,0.000049,0.237556,0.942051
5151001,-0.273953,-0.299094,0.299251,-0.574909,-0.518644,0.143146,0.373781,-0.261325,-0.230046,-0.583551,...,1.499954,1.489652,2.214694,2.006850,1.885379,2.309070,1.841269,0.164359,0.450599,1.217615
4500600,-0.532940,-0.392493,-0.047658,-0.809060,-0.666268,-0.166517,-0.108198,-0.429701,-0.539672,-0.846786,...,0.740231,0.696821,1.952500,1.534140,1.557593,1.826277,1.748809,-0.441402,-0.116242,0.601394


In [16]:
# unscored origins are ones with no sampled units, expected at small sample_frac
print(f"origins with no units: {(~scored).sum()} / {n_origins}")
print(f"predicted stay share: {stay_sum.sum() / perwt_sum.sum():.4f}")
print(
    f"observed stay share:   {(df['PERWT'] * df['STAY']).sum() / df['PERWT'].sum():.4f}"
)
print(f"predicted movers: {flow_matrix.to_numpy().sum():,.0f}")

origins with no units: 0 / 975
predicted stay share: 0.9422
observed stay share:   0.9414
predicted movers: 10,386,590


### Predicted vs. observed flows

The sampled units' own moves, on the same MIGPUMA x PUMA grid. At small `sample_frac` the observed
side is far sparser than the predicted one, so this is a shape check, not a fit statistic.

In [17]:
movers = df[df["STAY"] == 0]
observed = (
    movers.groupby(["ORIGIN", "CHOSEN"])["PERWT"]
    .sum()
    .unstack()
    .reindex(index=origin_migpumas, columns=dest_pumas)
    .fillna(0.0)
    * scale_up
)

difference = flow_matrix - observed
print(f"RMSE over the grid: {np.sqrt((difference**2).to_numpy().mean()):,.2f}")
difference.abs().sum(axis=1).sort_values(ascending=False).head(10)

RMSE over the grid: 133.30


0603700    290616.402227
3603800    168767.743251
3604000    163389.411903
0607300    150512.342312
1703400    149825.975090
3604100    145771.648051
2500390    137608.186218
0605900    123279.356412
3603200    116635.490819
4802300    116523.301041
dtype: float64

In [18]:
out_dir = Path("results")
tag = f"{Path(pums_file).stem}_{Path(spec_file).stem}"
utility_matrix.to_parquet(out_dir / f"utility_matrix_{tag}.parquet")
flow_matrix.to_parquet(out_dir / f"flow_matrix_{tag}.parquet")